# Northstar Employee Silver → Gold

Transforms validated Silver employee records into business-ready Gold
datasets for reporting, dashboards, and machine learning.

In [0]:
%python
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Databricks Repos normally adds the repository root to sys.path.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.paths import PATHS

print(PATHS)
print(dir(PATHS))

In [0]:
%python
GOLD_ROOT = (
    f"abfss://gold@{PATHS.storage_account}.dfs.core.windows.net/northstar"
)

GOLD_EMPLOYEE_PATH = f"{GOLD_ROOT}/employee_summary"
GOLD_DEPARTMENT_PATH = f"{GOLD_ROOT}/department_summary"
GOLD_EMPLOYER_PATH = f"{GOLD_ROOT}/employer_summary"
GOLD_STATE_PATH = f"{GOLD_ROOT}/state_summary"
GOLD_EXECUTIVE_KPI_PATH = f"{GOLD_ROOT}/executive_kpis"

print(f"Silver path: {SILVER_PATH}")
print(f"Gold employee path: {GOLD_EMPLOYEE_PATH}")
print(f"Gold department path: {GOLD_DEPARTMENT_PATH}")
print(f"Gold employer path: {GOLD_EMPLOYER_PATH}")
print(f"Gold state path: {GOLD_STATE_PATH}")
print(f"Gold executive KPI path: {GOLD_EXECUTIVE_KPI_PATH}")

In [0]:
%python
silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

silver_rows = silver_df.count()

print(f"Silver rows read: {silver_rows:,}")

In [0]:
%python
silver_df.printSchema()

In [0]:
%python
display(silver_df.limit(10))

as_of_date = F.to_date(F.lit(BUSINESS_DATE))

employer_summary_df = (
    silver_df
    .groupBy("employer_id")
    .agg(
        F.count("*").alias("total_employees"),

        F.sum(
            F.when(F.col("employment_status") == "Active", 1).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(F.col("employment_status") == "Terminated", 1).otherwise(0)
        ).alias("terminated_employees"),

        F.sum(
            F.when(F.col("coverage_eligible_as_of_date") == True, 1).otherwise(0)
        ).alias("coverage_eligible_employees"),

        F.round(F.avg("employee_age"), 2).alias("average_employee_age"),

        F.round(F.avg("years_of_service"), 2).alias(
            "average_years_of_service"
        ),

        F.countDistinct("department").alias("department_count"),

        F.countDistinct("state").alias("state_count"),
    )
    .withColumn(
        "active_employee_pct",
        F.round(
            F.col("active_employees") /
            F.col("total_employees") * 100,
            2,
        ),
    )
    .withColumn(
        "coverage_eligible_pct",
        F.round(
            F.col("coverage_eligible_employees") /
            F.col("total_employees") * 100,
            2,
        ),
    )
    .withColumn("business_date", as_of_date)
    .withColumn("processing_timestamp", F.current_timestamp())
)

display(
    employer_summary_df.orderBy(
        F.desc("total_employees")
    )
)

In [0]:
%python
(
    employer_summary_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(GOLD_EMPLOYER_PATH)
)

employer_gold_count = (
    spark.read
    .format("delta")
    .load(GOLD_EMPLOYER_PATH)
    .count()
)

print(f"Employer Gold rows written: {employer_gold_count:,}")

if employer_gold_count != employer_summary_df.count():
    raise RuntimeError(
        "Employer Gold validation failed: "
        f"expected={employer_summary_df.count():,}, "
        f"actual={employer_gold_count:,}"
    )

In [0]:
%python
department_summary_df = (
    silver_df
    .groupBy(
        "employer_id",
        "department",
    )
    .agg(
        F.count("*").alias("total_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Active",
                1,
            ).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Terminated",
                1,
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.sum(
            F.when(
                F.col("coverage_eligible_as_of_date") == True,
                1,
            ).otherwise(0)
        ).alias("coverage_eligible_employees"),

        F.round(
            F.avg("employee_age"),
            2,
        ).alias("average_employee_age"),

        F.round(
            F.avg("years_of_service"),
            2,
        ).alias("average_years_of_service"),

        F.countDistinct("state").alias("state_count"),
    )
    .withColumn(
        "active_employee_pct",
        F.round(
            F.col("active_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "termination_rate_pct",
        F.round(
            F.col("terminated_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "coverage_eligible_pct",
        F.round(
            F.col("coverage_eligible_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "business_date",
        F.to_date(F.lit(BUSINESS_DATE)),
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp(),
    )
)

display(
    department_summary_df.orderBy(
        "employer_id",
        F.desc("total_employees"),
    )
)

In [0]:
%python
(
    department_summary_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(GOLD_DEPARTMENT_PATH)
)

department_gold_count = (
    spark.read
    .format("delta")
    .load(GOLD_DEPARTMENT_PATH)
    .count()
)

print(f"Department Gold rows written: {department_gold_count:,}")

if department_gold_count != department_summary_df.count():
    raise RuntimeError(
        "Department Gold validation failed: "
        f"expected={department_summary_df.count():,}, "
        f"actual={department_gold_count:,}"
    )

In [0]:
%python
state_summary_df = (
    silver_df
    .groupBy(
        "state"
    )
    .agg(
        F.count("*").alias("employee_count"),

        F.sum(
            F.when(
                F.col("employment_status")=="Active",
                1
            ).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status")=="Terminated",
                1
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.round(
            F.avg("employee_age"),
            2
        ).alias("average_age"),

        F.round(
            F.avg("years_of_service"),
            2
        ).alias("average_service")
    )
    .withColumn(
        "termination_rate_pct",
        F.round(
            F.col("terminated_employees")
            /
            F.col("employee_count")
            *100,
            2
        )
    )
)

state_summary_df = (
    silver_df
    .groupBy("state")
    .agg(
        F.count("*").alias("employee_count"),

        F.sum(
            F.when(
                F.col("employment_status") == "Active",
                1,
            ).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Terminated",
                1,
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.sum(
            F.when(
                F.col("coverage_eligible_as_of_date") == True,
                1,
            ).otherwise(0)
        ).alias("coverage_eligible_employees"),

        F.round(
            F.avg("employee_age"),
            2,
        ).alias("average_age"),

        F.round(
            F.avg("years_of_service"),
            2,
        ).alias("average_service"),

        F.countDistinct("employer_id").alias("employer_count"),

        F.countDistinct("department").alias("department_count"),
    )
    .withColumn(
        "active_employee_pct",
        F.round(
            F.col("active_employees")
            / F.col("employee_count")
            * 100,
            2,
        ),
    )
    .withColumn(
        "termination_rate_pct",
        F.round(
            F.col("terminated_employees")
            / F.col("employee_count")
            * 100,
            2,
        ),
    )
    .withColumn(
        "coverage_eligible_pct",
        F.round(
            F.col("coverage_eligible_employees")
            / F.col("employee_count")
            * 100,
            2,
        ),
    )
    .withColumn(
        "business_date",
        F.to_date(F.lit(BUSINESS_DATE)),
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp(),
    )
)

display(
    state_summary_df.orderBy(
        F.desc("employee_count")
    )
)

In [0]:
%python
executive_kpi_df = (
    silver_df
    .agg(
        F.count("*").alias("total_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Active",
                1,
            ).otherwise(0)
        ).alias("active_employees"),

        F.sum(
            F.when(
                F.col("employment_status") == "Terminated",
                1,
            ).otherwise(0)
        ).alias("terminated_employees"),

        F.sum(
            F.when(
                F.col("coverage_eligible_as_of_date") == True,
                1,
            ).otherwise(0)
        ).alias("coverage_eligible_employees"),

        F.countDistinct("employer_id").alias("employer_count"),
        F.countDistinct("department").alias("department_count"),
        F.countDistinct("state").alias("state_count"),

        F.round(
            F.avg("employee_age"),
            2,
        ).alias("average_employee_age"),

        F.round(
            F.avg("years_of_service"),
            2,
        ).alias("average_years_of_service"),
    )
    .withColumn(
        "active_employee_pct",
        F.round(
            F.col("active_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "termination_rate_pct",
        F.round(
            F.col("terminated_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "coverage_eligible_pct",
        F.round(
            F.col("coverage_eligible_employees")
            / F.col("total_employees")
            * 100,
            2,
        ),
    )
    .withColumn(
        "business_date",
        F.to_date(F.lit(BUSINESS_DATE)),
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp(),
    )
)

display(executive_kpi_df)

In [0]:
%python
(
    executive_kpi_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(GOLD_EXECUTIVE_KPI_PATH)
)

executive_kpi_count = (
    spark.read
    .format("delta")
    .load(GOLD_EXECUTIVE_KPI_PATH)
    .count()
)

if executive_kpi_count != 1:
    raise RuntimeError(
        "Executive KPI Gold validation failed: "
        f"expected=1, actual={executive_kpi_count:,}"
    )

print("Executive KPI Gold rows written: 1")